# Feature Family Combination Brute Force (2, 3, 4)

This notebook evaluates every feature-family combination of size **2**, then **3**, then **4** using a consistent train/test split and logs metrics for each run.

In [ ]:
from __future__ import annotations

import itertools
import json
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tqdm.auto import tqdm

In [ ]:
def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("Could not locate repository root (missing pyproject.toml).")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
PIPELINE_DIR = REPO_ROOT / "feature_extraction" / "pipelines"
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

from pipeline_common import (
    DEFAULT_ALL_FAMILIES,
    feature_columns_from_families,
    load_base_metadata,
    load_family_frames,
    merge_feature_families,
)

COMBINATION_SIZES = (2, 3, 4)
FAMILIES = DEFAULT_ALL_FAMILIES.copy()
INCLUDE_XXX = False
REQUIRE_AGREEMENT = True
EXCLUDED_EMOTIONS = ("sur", "fea", "oth", "dis")

RANDOM_STATE = 42
TEST_SIZE = 0.20
MODEL_MAX_ITER = 2000

OUT_DIR = REPO_ROOT / "feature_test" / "classical_feature_system" / "artifacts" / "family_combo_bruteforce"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root: {REPO_ROOT}")
print(f"Output dir: {OUT_DIR}")
print(f"Requested families: {len(FAMILIES)}")
print(f"Combination sizes: {COMBINATION_SIZES}")

In [ ]:
base_df = load_base_metadata(
    REPO_ROOT,
    include_xxx=INCLUDE_XXX,
    require_agreement=REQUIRE_AGREEMENT,
    excluded_emotions=EXCLUDED_EMOTIONS,
)

family_frames, family_report = load_family_frames(
    REPO_ROOT,
    families=FAMILIES,
    prefix_features=True,
)

report_df = pd.DataFrame(family_report).sort_values(["available", "family"], ascending=[False, True])
display(report_df)

available_families = [family for family in FAMILIES if family in family_frames]
if len(available_families) < min(COMBINATION_SIZES):
    raise ValueError(
        f"Need at least {min(COMBINATION_SIZES)} available families, got {len(available_families)}"
    )

merged_df = merge_feature_families(
    base_df,
    {family: family_frames[family] for family in available_families},
)
merged_df = merged_df.dropna(subset=["emotion"]).reset_index(drop=True)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(merged_df["emotion"].astype(str).to_numpy())
idx = np.arange(len(merged_df))

idx_train, idx_test = train_test_split(
    idx,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

y_train = y[idx_train]
y_test = y[idx_test]

print(f"Rows in merged set: {len(merged_df):,}")
print(f"Available families: {len(available_families)} -> {available_families}")
print(f"Classes: {list(label_encoder.classes_)}")
print(f"Train rows: {len(idx_train):,} | Test rows: {len(idx_test):,}")

In [ ]:
combo_plan_by_size: dict[int, list[tuple[str, ...]]] = {}
for k in COMBINATION_SIZES:
    combos = list(itertools.combinations(available_families, k))
    combo_plan_by_size[k] = combos
    print(f"k={k}: {len(combos):,} combinations")

total_combinations = sum(len(v) for v in combo_plan_by_size.values())
print(f"Total combinations to evaluate: {total_combinations:,}")

In [ ]:
def build_model() -> Pipeline:
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value=0.0)),
            ("scaler", StandardScaler(with_mean=False)),
            (
                "clf",
                LogisticRegression(
                    solver="saga",
                    max_iter=MODEL_MAX_ITER,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


results: list[dict[str, object]] = []
for k in COMBINATION_SIZES:
    combos = combo_plan_by_size.get(k, [])
    if not combos:
        continue

    for combo in tqdm(combos, total=len(combos), desc=f"{k}-family combos"):
        feature_cols = feature_columns_from_families(merged_df, list(combo))
        if not feature_cols:
            continue

        x_train = merged_df.iloc[idx_train][feature_cols]
        x_test = merged_df.iloc[idx_test][feature_cols]

        model = build_model()
        started = time.perf_counter()
        model.fit(x_train, y_train)
        fit_seconds = time.perf_counter() - started

        y_pred = model.predict(x_test)

        results.append(
            {
                "combination_size": k,
                "families": "|".join(combo),
                "family_count": len(combo),
                "feature_count": len(feature_cols),
                "train_rows": int(len(idx_train)),
                "test_rows": int(len(idx_test)),
                "accuracy": float(accuracy_score(y_test, y_pred)),
                "f1_macro": float(f1_score(y_test, y_pred, average="macro")),
                "f1_weighted": float(f1_score(y_test, y_pred, average="weighted")),
                "fit_seconds": float(fit_seconds),
            }
        )

if not results:
    raise RuntimeError("No results were produced. Check family files and feature columns.")

results_df = (
    pd.DataFrame(results)
    .sort_values(["f1_macro", "accuracy"], ascending=False)
    .reset_index(drop=True)
)

display(results_df.head(20))

In [ ]:
run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
results_csv = OUT_DIR / f"combo_results_{run_id}.csv"
summary_json = OUT_DIR / f"combo_summary_{run_id}.json"

results_df.to_csv(results_csv, index=False)

best_row = results_df.iloc[0].to_dict()
summary = {
    "run_id": run_id,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "combination_sizes": list(COMBINATION_SIZES),
    "requested_families": FAMILIES,
    "available_families": available_families,
    "evaluated_combinations": int(len(results_df)),
    "model": "logistic_regression_saga",
    "model_max_iter": MODEL_MAX_ITER,
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "best_result": best_row,
}
summary_json.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(f"Saved results CSV: {results_csv}")
print(f"Saved summary JSON: {summary_json}")

size_summary = (
    results_df.groupby("combination_size")["f1_macro"]
    .agg(["count", "mean", "max"])
    .reset_index()
)
display(size_summary)